One problem maybe tokenizing 1000 samples takes about 12 minutes, so the whole dataset will tike 12 * 42 = 504 minutes

In [3]:
import pandas as pd
import torch
import transformers
from sklearn.model_selection import train_test_split

# Load and rename the column
file_path = '../data/raw/gender.csv'
df = pd.read_csv(file_path)
df.rename(columns={"auhtor_ID": "author_ID"}, inplace=True)
df = df.sample(100)

# Check the class distribution (female label)
print(df['female'].value_counts())

train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['female'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['female'])

print(f"\nTraining set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")

# Check class distribution of splits
print("\nTraining set label distribution:")
print(train_df['female'].value_counts())
print("\nValidation set label distribution:")
print(val_df['female'].value_counts())
print("\nTest set label distribution:")
print(test_df['female'].value_counts())

female
0    56
1    44
Name: count, dtype: int64

Training set size: 70
Validation set size: 15
Test set size: 15

Training set label distribution:
female
0    39
1    31
Name: count, dtype: int64

Validation set label distribution:
female
0    8
1    7
Name: count, dtype: int64

Test set label distribution:
female
0    9
1    6
Name: count, dtype: int64


In [20]:
import pandas as pd
import torch
from transformers import DistilBertTokenizerFast, DistilBertModel

# Initialize the tokenizer and model
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

# Tokenize the text data and create attention masks
def tokenize_data(dataframe):
    # Tokenize the text data
    tokenized = tokenizer(dataframe['post'].tolist(), padding=True, truncation=True, return_tensors='pt')


    print(tokenized)

    return tokenized

# Tokenize the data
train_tokenized = tokenize_data(train_df)


{'input_ids': tensor([[  101,  1997,  3182,  ...,  2342,  2009,   102],
        [  101, 20228, 12722,  ...,  1005,  2965,   102],
        [  101,  2005,  7908,  ...,  1012,  5024,   102],
        ...,
        [  101,  2031, 11085,  ..., 16454,  2121,   102],
        [  101,  2024,  2025,  ...,  2288,  2242,   102],
        [  101,  2757,  1045,  ...,  1037,  2267,   102]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])}


In [ ]:
from transformers import DistilBertModel, DistilBertTokenizer
from transformers import DistilBertTokenizerFast

# tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

# Freeze...
for param in model.parameters():
    param.requires_grad = False

# Function to generate embeddings using DistilBERT
def generate_embeddings(texts, batch_size=500000):
    all_embeddings = []

    # Process texts in batches
    for i in range(0, len(texts), batch_size):
        print('starting', i)
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
        print('tokenized', i)

        with torch.no_grad():
            print('embeddings start', i)
            outputs = model(**inputs)
            embeddings = outputs.last_hidden_state.mean(dim=1)
            all_embeddings.append(embeddings)
            print('embeddings done', i)

    return torch.cat(all_embeddings, dim=0)

# For now just cut off the posts text after 512 tokens
train_embeddings = generate_embeddings(train_df['post'].tolist())

starting 0
tokenized 0
embeddings start 0


In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Convert embeddings and labels to numpy arrays
X_train = train_embeddings.numpy()
y_train = train_df['female'].values

# Create and train the logistic regression model
classifier = LogisticRegression(max_iter=100)
classifier.fit(X_train, y_train)

# Evaluate the model
X_val = generate_embeddings(val_df['post'].tolist()).numpy()
y_val = val_df['female'].values
y_pred = classifier.predict(X_val)

# Calculate accuracy
print(classification_report(y_val, y_pred))

              precision    recall  f1-score   support

           0       0.74      0.81      0.78        79
           1       0.77      0.69      0.73        71

    accuracy                           0.75       150
   macro avg       0.75      0.75      0.75       150
weighted avg       0.75      0.75      0.75       150

